# Cell communication analysis: mcTECs

In [ ]:
import os
import sys
from datetime import datetime
today = datetime.today().strftime('%Y-%m-%d')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import anndata as ad
import hdf5plugin
#import liana as li

import warnings
warnings.filterwarnings('ignore', category=ad.ImplicitModificationWarning)

# Add repo path to sys path (allows to access scripts and metadata from repo)
repo_path,_ = os.path.split(os.path.split(os.getcwd())[0])
repo_path = '/nfs/team205/lm25/thymus_projects/thymus_ageing_atlas/General_analysis'
sys.path.insert(1, repo_path) 
sys.path.insert(2, '/nfs/team205/lm25/thymus_projects/thymus_ageing_atlas/General_analysis/scripts')

# Autoreload custom scripts
%load_ext autoreload
%autoreload 2

# Define paths
plots_path = f'{repo_path}/plots/'
data_path = f'{repo_path}/data/'
model_path = os.path.join(repo_path, 'models')
general_data_path = '/nfs/team205/lm25/thymus_projects/thymus_ageing_atlas/General_analysis/data'

print('Dir for plots: {}'.format(plots_path))
print('Dir for data: {}'.format(data_path))

# Formatting
from matplotlib import font_manager
font_manager.fontManager.addfont("/nfs/team205/ny1/ThymusSpatialAtlas/software/Arial.ttf")
plt.style.use('/nfs/team205/lm25/thymus_projects/thymus_ageing_atlas/General_analysis/scripts/plotting/thyAgeing.mplstyle')

# Import custom scripts
from utils import get_latest_version,update_obs,freq_by_donor
from anno_levels import get_ct_levels, get_ct_palette, age_group_levels, age_group_palette
from plotting.utils import plot_grouped_boxplot, calc_figsize

In [ ]:
# Define columns
col_cell_type_broad = 'taa_l3'
col_cell_type_fine = 'taa_l4'
# col_cell_type_broad_levels = get_ct_levels(col_cell_type_broad, taa_l1 = ['T', 'NK'])
# col_cell_type_fine_levels = get_ct_levels(col_cell_type_fine, taa_l1 = ['T', 'NK'])
col_age_group = 'age_group'
col_age_group_levels = eval(f'{col_age_group}_levels')

In [ ]:
ct_anno = pd.read_csv(f'{general_data_path}/objects/rna/thyAgeing_all_scvi_v5_2025-10-15_curatedAnno_v11.csv', index_col = 0)

mctec_barcodes = ct_anno[ct_anno['taa_l4'] == 'mcTEC'].index.tolist()

# Load adata
object_version = 'v5_2025-04-03'
adata = ad.read_h5ad(f'{general_data_path}/objects/rna/thyAgeing_all_scvi_{object_version}.zarr', backed = 'r')

# Subset adata to only include mcTEC cells
adata_mctec = adata[mctec_barcodes, :].to_memory()

# Add annotations to adata
adata_mctec.obs = adata_mctec.obs.join(ct_anno, how = 'left')
adata_mctec = adata_mctec[(adata_mctec.obs['anno_status'] == 'include') & (adata_mctec.obs['qc_status'] == 'PASS')]

# Update metadata
latest_meta_path = get_latest_version(dir = f'{general_data_path}/metadata', file_prefix='Thymus_ageing_metadata')
latest_meta = pd.read_excel(latest_meta_path)
update_obs(adata_mctec, latest_meta, on = 'index', ignore_warning = True)

adata_mctec.shape

Filter genes to receptors/ligands expressed in mcTEC:

In [ ]:
# Load CellPhoneDB database (v5.0.0)
ia_db = pd.read_csv(f'{data_path}/references/ccc_databases/cellphoneDB_v5.0.0_interaction_input.csv')
ia_db[['ligand', 'receptor']] = ia_db['interactors'].str.rsplit('-', n=1, expand=True)
ia_db['receptor'] = ia_db['receptor'].str.split('+')
ia_db = ia_db.explode('receptor')
ia_db.head()

In [ ]:
adata_mctec = adata_mctec[:, adata_mctec.var_names.isin(ia_db['ligand'].unique().tolist() + ia_db['receptor'].unique().tolist())]
adata_mctec.var['frac_expressed'] = np.array((adata_mctec.X > 0).sum(axis=0) / adata_mctec.n_obs).flatten()

adata_mctec

In [ ]:
plt.figure(figsize=calc_figsize(width=80, height=60))
sns.histplot(adata_mctec.var['frac_expressed'], bins=50, kde=False, color='steelblue')
plt.xlabel('Fraction expressed')
plt.ylabel('Number of genes')
plt.title('Histogram of frac_expressed in adata_mctec')
plt.tight_layout()
plt.show()

In [ ]:
lr_oi = adata_mctec.var_names[adata_mctec.var['frac_expressed'] > 0.1].tolist()
print(f'Number of ligands/receptors expressed in >10% of cells: {len(lr_oi)}')

ia_oi = ia_db[(ia_db['receptor'].isin(lr_oi)) & (ia_db['ligand'].isin(adata.var_names))]
print(f'Number of interactions with receptor expressed in >10% of cells: {ia_oi.shape[0]}')

all_lr = set(ia_oi['ligand'].unique().tolist() + ia_oi['receptor'].unique().tolist())
print(f'Total number of unique ligands/receptors in CellPhoneDB: {len(all_lr)}')

Identify cell types which could communicate with mcTEc based on niche assignments:

In [ ]:
niche_comp = pd.read_csv('/nfs/team205/vk8/projects/thymus_ageing_atlas/Spatial_analyses/results/xenium/mcTEC_niche_k6/mctec_niche_k6_taa_l3_comb_composition_pct.csv', index_col=0)
niche_recovery = pd.read_csv('/nfs/team205/vk8/projects/thymus_ageing_atlas/Spatial_analyses/results/xenium/mcTEC_niche_k6/mctec_niche_k6_per_donor_pct.csv', index_col=0)
niche_recovery['age_group'] = ['adult' if x.startswith('A') else 'paed' if x.startswith('Z') else 'infant' for x in niche_recovery.index]
niche_recovery

In [ ]:
niche_recovery.groupby('age_group').mean().T.plot(kind='bar', figsize=calc_figsize(width=80, height=60))

In [ ]:
niche_groups = niche_recovery.groupby('age_group').mean().T

pct_thres = 10
shared_niches = niche_groups.loc[(niche_groups['adult'] > pct_thres) & (niche_groups['paed'] > pct_thres) & (niche_groups['infant'] > pct_thres)].index.tolist()
adult_niches = [x for x in niche_groups.index if x not in shared_niches]

print(f'Number of shared niches: {len(shared_niches)}')
print(f'Number of adult niches: {len(adult_niches)}')

## Niches present in paed and adult

In [ ]:
# Proportion of niches in each age group
niche_ratios = niche_groups.loc[shared_niches, :]
niche_ratios = niche_ratios.div(niche_ratios.sum(axis=0), axis=1)
niche_ratios

In [ ]:
niche_comp_shared = niche_comp.loc[shared_niches, :] / 100
niche_comp_shared = pd.DataFrame(niche_ratios.values.T @ niche_comp_shared.values, columns=niche_comp_shared.columns, index=niche_ratios.columns)
niche_comp_shared

In [ ]:
ct_prop_threshold = 0.02

ct_per_niche = {a: niche_comp_shared.columns[niche_comp_shared.loc[a] >= ct_prop_threshold].tolist() for a in niche_comp_shared.index}
all_ct_oi = set([ct for ct_list in ct_per_niche.values() for ct in ct_list])

import pprint
pprint.pprint(ct_per_niche, compact=True, width=120)
pprint.pprint(all_ct_oi, compact=True, width=120)


In [ ]:
[c for c in all_ct_oi if c not in ct_anno['taa_l3'].values]

In [ ]:
# Subset adata
adata_sub_barcodes = ct_anno[ct_anno['taa_l3'].isin(all_ct_oi)].index.tolist()
adata_sub = adata[adata_sub_barcodes,adata.var_names.isin(all_lr)].to_memory()

# Add annotations to adata
adata_sub.obs = adata_sub.obs.join(ct_anno, how = 'left')
adata_sub = adata_sub[(adata_sub.obs['anno_status'] == 'include') & (adata_sub.obs['qc_status'] == 'PASS')]

# Update metadata
latest_meta_path = get_latest_version(dir = f'{general_data_path}/metadata', file_prefix='Thymus_ageing_metadata')
latest_meta = pd.read_excel(latest_meta_path)
update_obs(adata_sub, latest_meta, on = 'index', ignore_warning = True)

adata_sub.obs['cell_type'] = adata_sub.obs['taa_l3']

adata_sub.obs['cell_type'].value_counts()

Inlcude donors with 50 mcTECs

In [ ]:
# Filter out donors with fewer than 50 mcTECs
min_cells = 50
col_sample = 'donor'
ncells_by_sample = adata_sub[adata_sub.obs[col_cell_type_broad] == 'mcTEC'].obs[col_sample].value_counts().to_frame(name = 'n_cells')
print('Removing {} donors with fewer than {} cells'.format(ncells_by_sample[ncells_by_sample['n_cells'] < min_cells].shape[0], min_cells))
adata_sub = adata_sub[~adata_sub.obs[col_sample].isin(ncells_by_sample[ncells_by_sample['n_cells'] < min_cells].index)]

In [ ]:
# Level to cell type assignment
ct_levels = [c for c in get_ct_levels('taa_l3') if c in all_ct_oi]
import pprint
pprint.pprint(ct_levels, compact = True)

In [ ]:
# # Load global sample composition data
# taa_l3_freq = pd.read_csv(f'{data_path}/analyses/freqAnalysis/thyAgeing_all_{col_cell_type_broad}_byDonor_freq.csv', index_col = 0)
# taa_l3_freq = taa_l3_freq.loc[taa_l3_freq[col_cell_type_broad].isin(ct_levels),:].rename(columns = {col_cell_type_broad: 'cell_type'})

# # Calculate fraction of cells expressing each ligand/receptor in each cell type and donor
# ct_freq = ct_freq.merge(niche_comp_shared.melt(ignore_index = False, var_name='cell_type', value_name='niche_prop').reset_index(), on = ['cell_type', 'age_group'], how = 'left')
# ct_freq.fillna({'niche_prop': 0}, inplace = True)
# ct_freq['cell_prop'] = ct_freq['mean_prop'] * ct_freq['niche_prop']
# ct_freq.head()

In [ ]:
# Niche composition for each cell type by age group
ct_freq = niche_comp_shared.melt(ignore_index = False, var_name='cell_type', value_name='niche_prop').reset_index()
ct_freq = ct_freq.loc[ct_freq['cell_type'].isin(ct_levels), :]
ct_freq

In [ ]:
interactions_oi = ia_oi[['ligand', 'receptor']].drop_duplicates().apply(lambda x: (x['ligand'], x['receptor']), axis = 1).tolist()
interactions_oi 

In [ ]:
# Calculate fraction of cells expressing each ligand/receptor in each cell type and donor
adata_sub.obs['ct_donor'] = adata_sub.obs['donor'] + '__' + adata_sub.obs['cell_type']
ct_by_donor_dummy = pd.get_dummies(adata_sub.obs['ct_donor'])
n_cells_per_ct_donor = ct_by_donor_dummy.sum(axis=0)
X_bool = (adata_sub.X > 0).astype(int)

frac_by_ct = ct_by_donor_dummy.values.T * X_bool
frac_by_ct = frac_by_ct / n_cells_per_ct_donor.to_numpy()[:, None]

frac_by_ct

In [ ]:
frac_by_ct_df = pd.DataFrame(frac_by_ct, index=ct_by_donor_dummy.columns, columns=adata_sub.var_names)
frac_by_ct_df['cell_type'] = frac_by_ct_df.index.str.split('__').str[-1]
frac_by_ct_df['donor'] = frac_by_ct_df.index.str.split('__').str[0]
frac_by_ct_df = frac_by_ct_df.melt(id_vars=['cell_type', 'donor'], var_name='gene', value_name='frac_expressed')
frac_by_ct_df.head()

In [ ]:
lr_frac_expressed = ia_oi[['ligand', 'receptor']].drop_duplicates() \
    .merge(frac_by_ct_df, left_on='ligand', right_on='gene', how='left').rename(columns={'frac_expressed': 'ligand_props', 'cell_type' : 'source'}).drop(columns = 'gene') \
    .merge(frac_by_ct_df, left_on=['receptor', 'donor'], right_on=['gene', 'donor'], how='left').rename(columns={'frac_expressed': 'receptor_props', 'cell_type': 'target'}).drop(columns = 'gene') 
lr_frac_expressed['age_group'] = lr_frac_expressed['donor'].map(adata_sub.obs[['donor', 'age_group']].drop_duplicates().set_index('donor')['age_group'])
lr_frac_expressed['interaction'] = lr_frac_expressed['ligand'] + '->' + lr_frac_expressed['receptor']
lr_frac_expressed['cells'] = lr_frac_expressed['source'] + '->' + lr_frac_expressed['target']
lr_frac_expressed = lr_frac_expressed.merge(ct_freq[['age_group', 'cell_type', 'niche_prop']], left_on=['age_group', 'source'], right_on=['age_group', 'cell_type'], how='left').rename(columns={'niche_prop': 'source_prop'}).drop(columns=['cell_type']) \
    .merge(ct_freq[['age_group', 'cell_type','niche_prop']], left_on=['age_group', 'target'], right_on=['age_group', 'cell_type'], how='left').rename(columns={'niche_prop': 'target_prop'}).drop(columns=['cell_type'])

lr_frac_expressed

In [ ]:
# Subset to interactions with mcTEC as target
lr_frac_expressed = lr_frac_expressed[lr_frac_expressed['target'] == 'mcTEC'].dropna()
lr_frac_expressed.shape

In [ ]:
#lr_frac_expressed.to_csv(f'{data_path}/analyses/cellComm/mcTEC/shared_niche_lr_expressed_by_interaction_and_celltype.csv', index=False)

Replace source/target proportion with niche-specific proportions

In [ ]:
# Calculate mean properties (using all samples, not just TOT sort)
lr_frac_expressed['tot_prob'] = lr_frac_expressed['ligand_props'] * lr_frac_expressed['receptor_props'] * lr_frac_expressed['source_prop'] * lr_frac_expressed['target_prop']
mean_props = lr_frac_expressed.groupby(['interaction', 'cells', 'age_group']).agg({'ligand_props' : 'mean', 'receptor_props' : 'mean', 'tot_prob': 'mean'}).reset_index()
mean_props = mean_props.pivot(index = ['interaction', 'cells'], columns = 'age_group', values = ['ligand_props', 'receptor_props','tot_prob']).reset_index()
mean_props.columns = ['_'.join(col).strip() for col in mean_props.columns.values]
mean_props = mean_props.rename(columns = {'interaction_': 'interaction', 'cells_': 'cells'})
mean_props.head()

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from scipy.stats import ranksums
from statsmodels.stats.multitest import multipletests

# Filter interactions
# Select interactions with at least one age group with expression > 0.1
target_age_group = ['adult', 'paed']
min_expr = 0.1

# Keeping any interaction with at least one age group with expression > in expr
#ia_to_keep = mean_props.loc[mean_props[['ligand_props_' + ag for ag in target_age_group] + ['receptor_props_' + ag for ag in target_age_group]].ge(min_expr).all(axis=1)][['interaction', 'cells']]

# Keeping interactions where L and R are both expressed at min expr in the same age group
ia_to_keep = []
for g in target_age_group:
    ia_to_keep.append(mean_props.loc[mean_props[[f'ligand_props_{g}', f'receptor_props_{g}']].ge(min_expr).all(axis=1)][['interaction', 'cells']])
ia_to_keep = pd.concat(ia_to_keep, axis = 0).drop_duplicates()

print(f'Number of interactions to keep: {ia_to_keep.shape[0]}')

donor_n = adata.obs[['donor', col_age_group]].drop_duplicates().groupby('age_group').agg({'donor': 'count'}).to_dict()['donor']
# Define a function to process a single cell
def process_cells(ct):
    ias = ia_to_keep[ia_to_keep['cells'] == ct]['interaction'].unique()
    test_ranksum = []
    for ia in ias:
        test_data = lr_frac_expressed[(lr_frac_expressed['interaction'] == ia) & (lr_frac_expressed['cells'] == ct)][['donor', col_age_group, 'tot_prob']].dropna(subset=['tot_prob']).copy()
        group1 = test_data[test_data['age_group'] == target_age_group[0]]['tot_prob'].to_numpy()
        if group1.shape[0] < donor_n[target_age_group[0]]:
            group1 = np.concatenate((group1, np.repeat(0, repeats=donor_n[target_age_group[0]] - group1.shape[0])))
        group2 = test_data[test_data['age_group'] == target_age_group[1]]['tot_prob'].to_numpy()
        if group2.shape[0] < donor_n[target_age_group[1]]:
            group2 = np.concatenate((group2, np.repeat(0, repeats=donor_n[target_age_group[1]] - group2.shape[0])))
        test_res = ranksums(group1, group2)
        test_ranksum.append([ia, ct, test_res[0], test_res[1]])
    test_ranksum = pd.DataFrame(test_ranksum, columns=['interaction', 'cells', 'statistic', 'pvalue'])
    test_ranksum['padj'] = multipletests(test_ranksum['pvalue'], method='fdr_bh')[1]
    return test_ranksum

# Use ThreadPoolExecutor with 4 threads to process cells in parallel
with ThreadPoolExecutor(max_workers=4) as executor:
    results = list(executor.map(process_cells, ia_to_keep['cells'].unique()))

# Combine results
ranksums_res = pd.concat(results, axis=0)
ranksums_res.to_csv(f'{data_path}/analyses/cellComm/mcTEC/shared_niche_{target_age_group[1]}_vs_{target_age_group[0]}_ranksums.csv')

ranksums_res.sort_values('statistic', ascending=False)

In [ ]:
ranksums_res = pd.read_csv(f'{data_path}/analyses/cellComm/mcTEC/shared_niche_{target_age_group[1]}_vs_{target_age_group[0]}_ranksums.csv', index_col = 0)
ranksums_res[['source', 'target']] = ranksums_res['cells'].str.split('->', expand=True)
ranksums_res[['ligand', 'receptor']] = ranksums_res['interaction'].str.split('->', expand=True)
ranksums_res.sort_values('statistic', ascending=False)

In [ ]:
ranksums_res_filtered = ranksums_res[ranksums_res['padj'] < 0.05]
ranksums_res_filtered['ia_change_direction'] = np.where(ranksums_res_filtered['statistic'] > 0, 'up', 'down')
ranksums_res_filtered.sort_values('statistic', ascending=False)

In [ ]:
ranksums_res_filtered['source'].value_counts()

In [ ]:
# Adult vs Infant
pprint.pprint(ranksums_res_filtered.groupby(['source','ia_change_direction'])['interaction'].agg(list).to_dict(), compact=True, width=120)

In [ ]:
# Paed vs Infant
pprint.pprint(ranksums_res_filtered.groupby(['source','ia_change_direction'])['interaction'].agg(list).to_dict(), compact=True, width=120)

In [ ]:
# Adult vs paed
pprint.pprint(ranksums_res_filtered.groupby(['source','ia_change_direction'])['interaction'].agg(list).to_dict(), compact=True, width=120)

This dataset is the complementary half of the story — and it changes the picture in an important way. The first analysis showed the adult niche losing signals; this one shows what's being **gained** by mcTEC as a receiver, from Fb-perilo, cTEC, mTECI, and B_plasma. Together, these two datasets suggest the adult mcTEC niche isn't a signaling vacuum — it's an actively rewired, different microenvironment, where trophic/structural investment is being **redirected toward mcTEC and away from cTEC**, not simply extinguished.

### 1. ECM–integrin signaling is redirected, not lost

The collagen/laminin/fibronectin/thrombospondin → integrin axis that disappeared from Fb-perilo/EC-cap→cTEC in your first dataset reappears here, amplified, as Fb-perilo→mcTEC, cTEC→mcTEC, and mTECI→mcTEC (COL4, COL6, COL16-18, COL22, COL27, LAMA3, LAMC1, FN1, THBS1, AGRN→ITGA2/3/6/AV/B1/B5, PTPRS). This strongly supports the reallocation model over a simple decay model: the cortical ECM/basement-membrane instructive network hasn't disappeared, it's been rebuilt around mcTEC instead of cTEC.

### 2. Notch2 ligation gain — a strong mechanistic candidate for the mTEC-biased, cTEC-blind phenotype

JAG1→NOTCH2/CD46/VASN, JAG2→NOTCH2/VASN, and DLL1→NOTCH2 all increase toward mcTEC, from Fb-perilo, cTEC, and mTECI simultaneously. This maps very well onto your phenotype. A 2020 fate-mapping study found that canonical Notch signaling maintains the early/primitive thymic epithelial progenitor state and specifically drives emergence of the medullary epithelial sublineage, implicating Notch as a switch that keeps progenitors undifferentiated while biasing what differentiation potential remains toward mTEC rather than cTEC (*Development*, 2020, "Canonical Notch signaling controls the early thymic epithelial progenitor cell state and emergence of the medullary epithelial lineage"). If Notch2 ligation is being reinforced on mcTECs in the adult niche, that's a plausible, testable explanation for exactly the asymmetric potential you're seeing — stalled/primitive state, cTEC potential lost, mTEC potential preserved.

### 3. TGFβ/BMP gain — active growth arrest, not just loss of pro-differentiation cues

TGFB1→ITGAV/TGFBR2/3 and TGFB3→TGFBR2/3 increase from Fb-perilo, cTEC, and mTECI toward mcTEC; BMP6→BMPR1A/BMPR2 increases from B_plasma. TGFβ is classically cytostatic and pro-fibrotic. This reframes the mechanism slightly from your first dataset: it's not only that instructive signals are withdrawn from would-be cTECs, but that mcTECs are actively receiving a growth-arrest signal. That's consistent with "stalled" rather than merely "understimulated."

### 4. FGF7/FGF10→FGFR2 — a genuine puzzle worth flagging

Fb-perilo→mcTEC gains FGF7→FGFR2 and FGF10→FGFR2. This is the textbook mesenchyme-derived TEC-proliferation axis — FGF7/FGF10 signaling through FGFR2IIIb on TEC is essential for TEC proliferation in thymus organogenesis, and its loss produces a hypoplastic, proliferation-arrested thymus (Revest et al. 2001, *J Immunol*; reviewed in Lopes/Non-Epithelial Stromal Cells review, *Front Immunol* 2021). That review also notes something relevant to your data: mesenchyme/FGF7/10 signaling is required for TEC *proliferation* but is largely dispensable for TEC *differentiation* — proliferation and differentiation are separable outputs of this axis.

Given that you describe mcTECs as low-proliferation, the gain of FGF7/10→FGFR2 signaling here is a genuine tension worth investigating rather than glossing over: either (a) the ligand is present but the downstream MAPK/proliferative response is uncoupled (worth checking pERK, DUSP/Sprouty negative feedback expression in mcTEC), or (b) the concurrent TGFβ/Notch gain is actively overriding the proliferative FGF signal. Either way, this looks like a genuinely informative point to dig into rather than something the CCC framework alone will resolve.

### 5. Retinoic acid receptor engagement — relevant to your fibroblast ALDH question

ALDH1A1→CRABP2, ALDH1A1→RARA, ALDH1A1→RXRA, ALDH1A1→RXRB all gain, now from **both** Fb-perilo and mTECI, directed at mcTEC. This is a meaningfully different observation from simply "ALDH1A1 goes up" — it shows the cognate RA-receptor/transport machinery (RARA, RXRA/B, CRABP2) engaging concordantly on the receiving compartment, which is closer to positive evidence for functional RA signaling than ALDH expression alone would be, since CCC scoring here requires receptor co-expression on the receiver, not just ligand expression on the sender. It doesn't replace the Rarb/Cyp26 concordance check you're planning, but it's a supportive, independent line of evidence pointing toward genuine RA-pathway engagement rather than purely oxidative-stress-driven ALDH activity — worth citing alongside your target-gene validation.

### 6. TRAIL/TWEAK death-receptor and decoy-receptor gain — candidate apoptosis-resistance signature

TNFSF10→TNFRSF10B (agonist, DR5) *and* TNFSF10→TNFRSF10D (decoy, DcR2) both increase, from nearly every sender (B_plasma, Fb-perilo, cTEC, mTECI); TNFSF12→TNFRSF12A (TWEAK–Fn14) also increases from Fb-perilo and mTECI. Co-upregulation of an apoptotic ligand alongside its decoy receptor on the same target is a recognized feature of senescent/stress-resistant epithelium — cells exposed to death-ligand pressure that survive via decoy receptor expression. This is a plausible, checkable explanation for how "non-functional" mcTECs persist and accumulate rather than being cleared: worth a spot IF check for DcR2/TNFRSF10D and canonical senescence markers (p16^INK4a, p21) in the mcTEC compartment.

### 7. Junctional reinforcement and compartment boundary signaling

Desmosomal (DSC2–DSG2, DSG2–DSC3), adherens (CDH1, CDH3), gap junction (GJA1–GJA1), and adhesion (CADM1, JAM3) signaling all increase from cTEC and mTECI toward mcTEC, alongside a broad gain in Eph–ephrin signaling (EFNA1, EFNB1, EFNB2 → EPHA1/2/4, EPHB6). Eph–ephrin signaling is a classic tissue-boundary-setting mechanism. Together these look like active construction of a bounded, cohesive replacement microdomain — mechanistic support for your description of mcTECs "forming their own niche" rather than simply drifting as disorganized residual cells.

### Synthesis

Read together with your first dataset, the picture becomes: cTECs are cut off from thymocyte crosstalk and from fibroblast/endothelial ECM and trophic support, while that same instructive repertoire (ECM-integrin, Notch2, TGFβ, FGF, RA-receptor engagement, junctional adhesion) is reallocated onto mcTEC. This is a reallocation/niche-replacement model, not a simple involution-by-attrition model — and it gives you several concrete, mechanistically motivated candidates (JAG1/JAG2–NOTCH2, TGFB1/3–TGFBR2/3, TNFSF10–TNFRSF10D) to prioritize for spatial/functional validation ahead of the RA-signaling and AP-1-regulon work you're already planning.

**Caveats carried over from before:** the same database-annotation noise applies (HSD17B12, SRD5A3, LIPA, PPIA are unlikely genuine paracrine ligands); this is still correlative, cross-sectional data, so "redirection" is a model to test, not a proven mechanism; and it's worth noting EC-cap doesn't appear as a sender in this "up" list at all — if that's not an artifact of how the comparison was run, it would suggest the ECM-integrin gain toward mcTEC is fibroblast/epithelial-driven specifically, and endothelium's contribution to the cortical niche is being lost outright rather than redirected, which would be worth stating explicitly in the paper.

## Niches present in mostly adults

In [ ]:
# Proportion of niches in each age group
niche_ratios = niche_groups.loc[adult_niches, :]
niche_ratios = niche_ratios.div(niche_ratios.sum(axis=0), axis=1)
niche_ratios

In [ ]:
niche_comp_adult= niche_comp.loc[adult_niches, :] / 100
niche_comp_adult = pd.DataFrame(niche_ratios.values.T @ niche_comp_adult.values, columns=niche_comp_adult.columns, index=niche_ratios.columns)
niche_comp_adult

In [ ]:
ct_prop_threshold = 0.02

ct_per_niche = {a: niche_comp_adult.columns[niche_comp_adult.loc[a] >= ct_prop_threshold].tolist() for a in niche_comp_adult.index}
all_ct_oi = set([ct for ct_list in ct_per_niche.values() for ct in ct_list])

import pprint
pprint.pprint(ct_per_niche, compact=True, width=120)
pprint.pprint(all_ct_oi, compact=True, width=120)


In [ ]:
# Niche composition for each cell type by age group
ct_freq = niche_comp_adult.melt(ignore_index = False, var_name='cell_type', value_name='niche_prop').reset_index()
ct_freq = ct_freq.loc[ct_freq['cell_type'].isin(all_ct_oi), :]
ct_freq

In [ ]:
# Subset adata
adata_sub_barcodes = ct_anno[ct_anno['taa_l3'].isin(all_ct_oi)].index.tolist()
adata_sub = adata[adata_sub_barcodes,adata.var_names.isin(all_lr)].to_memory()

# Add annotations to adata
adata_sub.obs = adata_sub.obs.join(ct_anno, how = 'left')
adata_sub = adata_sub[(adata_sub.obs['anno_status'] == 'include') & (adata_sub.obs['qc_status'] == 'PASS')]

# Update metadata
latest_meta_path = get_latest_version(dir = f'{general_data_path}/metadata', file_prefix='Thymus_ageing_metadata')
latest_meta = pd.read_excel(latest_meta_path)
update_obs(adata_sub, latest_meta, on = 'index', ignore_warning = True)

adata_sub.obs['cell_type'] = adata_sub.obs['taa_l3']

adata_sub.obs['cell_type'].value_counts()

In [ ]:
# Filter out donors with fewer than 50 mcTECs
min_cells = 50
col_sample = 'donor'
ncells_by_sample = adata_sub[adata_sub.obs[col_cell_type_broad] == 'mcTEC'].obs[col_sample].value_counts().to_frame(name = 'n_cells')
print('Removing {} donors with fewer than {} cells'.format(ncells_by_sample[ncells_by_sample['n_cells'] < min_cells].shape[0], min_cells))
adata_sub = adata_sub[~adata_sub.obs[col_sample].isin(ncells_by_sample[ncells_by_sample['n_cells'] < min_cells].index)]

In [ ]:
# Level to cell type assignment
ct_levels = [c for c in get_ct_levels('taa_l3') if c in all_ct_oi]
import pprint
pprint.pprint(ct_levels, compact = True)

In [ ]:
# Calculate fraction of cells expressing each ligand/receptor in each cell type and donor
adata_sub.obs['ct_donor'] = adata_sub.obs['donor'] + '__' + adata_sub.obs['cell_type']
ct_by_donor_dummy = pd.get_dummies(adata_sub.obs['ct_donor'])
n_cells_per_ct_donor = ct_by_donor_dummy.sum(axis=0)
X_bool = (adata_sub.X > 0).astype(int)

frac_by_ct = ct_by_donor_dummy.values.T * X_bool
frac_by_ct = frac_by_ct / n_cells_per_ct_donor.to_numpy()[:, None]

frac_by_ct_df = pd.DataFrame(frac_by_ct, index=ct_by_donor_dummy.columns, columns=adata_sub.var_names)
frac_by_ct_df['cell_type'] = frac_by_ct_df.index.str.split('__').str[-1]
frac_by_ct_df['donor'] = frac_by_ct_df.index.str.split('__').str[0]
frac_by_ct_df = frac_by_ct_df.melt(id_vars=['cell_type', 'donor'], var_name='gene', value_name='frac_expressed')
frac_by_ct_df.head()

In [ ]:
lr_frac_expressed = ia_oi[['ligand', 'receptor']].drop_duplicates() \
    .merge(frac_by_ct_df, left_on='ligand', right_on='gene', how='left').rename(columns={'frac_expressed': 'ligand_props', 'cell_type' : 'source'}).drop(columns = 'gene') \
    .merge(frac_by_ct_df, left_on=['receptor', 'donor'], right_on=['gene', 'donor'], how='left').rename(columns={'frac_expressed': 'receptor_props', 'cell_type': 'target'}).drop(columns = 'gene') 
lr_frac_expressed['age_group'] = lr_frac_expressed['donor'].map(adata_sub.obs[['donor', 'age_group']].drop_duplicates().set_index('donor')['age_group'])
lr_frac_expressed['interaction'] = lr_frac_expressed['ligand'] + '->' + lr_frac_expressed['receptor']
lr_frac_expressed['cells'] = lr_frac_expressed['source'] + '->' + lr_frac_expressed['target']
lr_frac_expressed = lr_frac_expressed.merge(ct_freq[['age_group', 'cell_type', 'niche_prop']], left_on=['age_group', 'source'], right_on=['age_group', 'cell_type'], how='left').rename(columns={'niche_prop': 'source_prop'}).drop(columns=['cell_type']) \
    .merge(ct_freq[['age_group', 'cell_type','niche_prop']], left_on=['age_group', 'target'], right_on=['age_group', 'cell_type'], how='left').rename(columns={'niche_prop': 'target_prop'}).drop(columns=['cell_type'])

lr_frac_expressed

In [ ]:
# Subset to interactions with mcTEC as target
lr_frac_expressed = lr_frac_expressed[lr_frac_expressed['target'] == 'mcTEC'].dropna()
lr_frac_expressed.shape

In [ ]:
# Calculate mean properties (using all samples, not just TOT sort)
lr_frac_expressed['tot_prob'] = lr_frac_expressed['ligand_props'] * lr_frac_expressed['receptor_props'] * lr_frac_expressed['source_prop'] * lr_frac_expressed['target_prop']
mean_props = lr_frac_expressed.groupby(['interaction', 'cells', 'age_group']).agg({'ligand_props' : 'mean', 'receptor_props' : 'mean', 'tot_prob': 'mean'}).reset_index()
mean_props = mean_props.pivot(index = ['interaction', 'cells'], columns = 'age_group', values = ['ligand_props', 'receptor_props','tot_prob']).reset_index()
mean_props.columns = ['_'.join(col).strip() for col in mean_props.columns.values]
mean_props = mean_props.rename(columns = {'interaction_': 'interaction', 'cells_': 'cells'})
mean_props.head()

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from scipy.stats import ranksums
from statsmodels.stats.multitest import multipletests

# Filter interactions
# Select interactions with at least one age group with expression > 0.1
target_age_group = ['paed', 'infant']
min_expr = 0.1

# Keeping any interaction with at least one age group with expression > in expr
#ia_to_keep = mean_props.loc[mean_props[['ligand_props_' + ag for ag in target_age_group] + ['receptor_props_' + ag for ag in target_age_group]].ge(min_expr).all(axis=1)][['interaction', 'cells']]

# Keeping interactions where L and R are both expressed at min expr in the same age group
ia_to_keep = []
for g in target_age_group:
    ia_to_keep.append(mean_props.loc[mean_props[[f'ligand_props_{g}', f'receptor_props_{g}']].ge(min_expr).all(axis=1)][['interaction', 'cells']])
ia_to_keep = pd.concat(ia_to_keep, axis = 0).drop_duplicates()

print(f'Number of interactions to keep: {ia_to_keep.shape[0]}')

donor_n = adata.obs[['donor', col_age_group]].drop_duplicates().groupby('age_group').agg({'donor': 'count'}).to_dict()['donor']
# Define a function to process a single cell
def process_cells(ct):
    ias = ia_to_keep[ia_to_keep['cells'] == ct]['interaction'].unique()
    test_ranksum = []
    for ia in ias:
        test_data = lr_frac_expressed[(lr_frac_expressed['interaction'] == ia) & (lr_frac_expressed['cells'] == ct)][['donor', col_age_group, 'tot_prob']].dropna(subset=['tot_prob']).copy()
        group1 = test_data[test_data['age_group'] == target_age_group[0]]['tot_prob'].to_numpy()
        if group1.shape[0] < donor_n[target_age_group[0]]:
            group1 = np.concatenate((group1, np.repeat(0, repeats=donor_n[target_age_group[0]] - group1.shape[0])))
        group2 = test_data[test_data['age_group'] == target_age_group[1]]['tot_prob'].to_numpy()
        if group2.shape[0] < donor_n[target_age_group[1]]:
            group2 = np.concatenate((group2, np.repeat(0, repeats=donor_n[target_age_group[1]] - group2.shape[0])))
        test_res = ranksums(group1, group2)
        test_ranksum.append([ia, ct, test_res[0], test_res[1]])
    test_ranksum = pd.DataFrame(test_ranksum, columns=['interaction', 'cells', 'statistic', 'pvalue'])
    test_ranksum['padj'] = multipletests(test_ranksum['pvalue'], method='fdr_bh')[1]
    return test_ranksum

# Use ThreadPoolExecutor with 4 threads to process cells in parallel
with ThreadPoolExecutor(max_workers=4) as executor:
    results = list(executor.map(process_cells, ia_to_keep['cells'].unique()))

# Combine results
ranksums_res = pd.concat(results, axis=0)
ranksums_res.to_csv(f'{data_path}/analyses/cellComm/mcTEC/adult_niche_{target_age_group[1]}_vs_{target_age_group[0]}_ranksums.csv')

ranksums_res.sort_values('statistic', ascending=False)

In [ ]:
ranksums_res = pd.read_csv(f'{data_path}/analyses/cellComm/mcTEC/adult_niche_{target_age_group[1]}_vs_{target_age_group[0]}_ranksums.csv', index_col = 0)
ranksums_res[['source', 'target']] = ranksums_res['cells'].str.split('->', expand=True)
ranksums_res[['ligand', 'receptor']] = ranksums_res['interaction'].str.split('->', expand=True)

ranksums_res_filtered = ranksums_res[ranksums_res['padj'] < 0.05]
ranksums_res_filtered['ia_change_direction'] = np.where(ranksums_res_filtered['statistic'] > 0, 'up', 'down')
ranksums_res_filtered.sort_values('statistic', ascending=False)

In [ ]:
# Adult vs Infant
pprint.pprint(ranksums_res_filtered.groupby(['source','ia_change_direction'])['interaction'].agg(list).to_dict(), compact=True, width=120)

In [ ]:
# Paed vs Infant
pprint.pprint(ranksums_res_filtered.groupby(['source','ia_change_direction'])['interaction'].agg(list).to_dict(), compact=True, width=120)

In [ ]:
# Adult vs paed
pprint.pprint(ranksums_res_filtered.groupby(['source','ia_change_direction'])['interaction'].agg(list).to_dict(), compact=True, width=120)